# Final Project

This project its a recoopilation of techinqiues learned during the course 02806 Social data analysis and visualization. 

> This markwdoen will cotain all sections of the final report, 

---
## 0 Setup

In [2]:
import pandas as pd
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE SETUP  (run once at top of notebook)
# Tufte / DAOST — high data-ink ratio, minimal chrome
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np

# ── Typography ─────────────────────────────────────────────────────────────────
plt.rcParams['font.family']           = 'sans-serif'
plt.rcParams['font.sans-serif']       = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus']    = False
plt.rcParams['font.size']             = 11          # base size everything scales from
plt.rcParams['axes.titlesize']        = 12
plt.rcParams['axes.titleweight']      = 'bold'
plt.rcParams['axes.titlepad']         = 10
plt.rcParams['axes.labelsize']        = 10
plt.rcParams['axes.labelcolor']       = '#333333'
plt.rcParams['xtick.labelsize']       = 9
plt.rcParams['ytick.labelsize']       = 9
plt.rcParams['legend.fontsize']       = 9
plt.rcParams['figure.titlesize']      = 14
plt.rcParams['figure.titleweight']    = 'normal'    # suptitle stays light

# ── Spines & Ticks ─────────────────────────────────────────────────────────────
plt.rcParams['axes.spines.top']       = False
plt.rcParams['axes.spines.right']     = False
plt.rcParams['axes.spines.left']      = True
plt.rcParams['axes.spines.bottom']    = True
plt.rcParams['axes.edgecolor']        = '#cccccc'   # soft spine colour
plt.rcParams['axes.linewidth']        = 0.8
plt.rcParams['xtick.color']           = '#555555'
plt.rcParams['ytick.color']           = '#555555'
plt.rcParams['xtick.direction']       = 'out'
plt.rcParams['ytick.direction']       = 'out'
plt.rcParams['xtick.major.size']      = 4
plt.rcParams['ytick.major.size']      = 4
plt.rcParams['xtick.major.width']     = 0.8
plt.rcParams['ytick.major.width']     = 0.8

# ── Grid ───────────────────────────────────────────────────────────────────────
plt.rcParams['axes.grid']             = True
plt.rcParams['grid.color']            = '#eeeeee'
plt.rcParams['grid.linewidth']        = 0.8
plt.rcParams['grid.linestyle']        = '-'
plt.rcParams['axes.axisbelow']        = True        # grid behind bars

# ── Figures ────────────────────────────────────────────────────────────────────
plt.rcParams['figure.facecolor']      = 'white'
plt.rcParams['axes.facecolor']        = 'white'
plt.rcParams['figure.dpi']            = 120
plt.rcParams['savefig.dpi']           = 150
plt.rcParams['savefig.bbox']          = 'tight'
plt.rcParams['savefig.facecolor']     = 'white'

# ── Legend ─────────────────────────────────────────────────────────────────────
plt.rcParams['legend.frameon']        = False       # no box
plt.rcParams['legend.loc']            = 'best'

# ── Semantic palette (reuse everywhere) ────────────────────────────────────────
GRADE_COLORS  = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}
DARK          = "#2c3e50"       # main data ink (lines, markers)
SPINE_COLOR   = "#cccccc"
TICK_COLOR    = "#555555"
BORO_ORDER    = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

In [3]:
# Inspection — Compare the 2019 and 2026 snapshots BEFORE merging.
# Goal: understand what each file contains, what's compatible, and what isn't.

import pandas as pd

# ── Load both ───────────────────────────────────────────────────────────────
df19 = pd.read_csv("data/NewYorkCity_Restaurant_Inspection_Results_2019.csv",
                   dtype={"ZIPCODE": str}, low_memory=False)
df26 = pd.read_csv("data/NewYorkCity_Restaurant_Inspection_Results_2026.csv",
                   dtype={"ZIPCODE": str}, low_memory=False)

# Parse dates uniformly
for d in (df19, df26):
    d["INSPECTION DATE"] = pd.to_datetime(d["INSPECTION DATE"], errors="coerce")

print("=" * 70)
print("BASIC SHAPE")
print("=" * 70)
print(f"2019 file: {len(df19):>9,} rows × {df19.shape[1]} columns")
print(f"2026 file: {len(df26):>9,} rows × {df26.shape[1]} columns")

# ── Column comparison ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COLUMN DIFFERENCES")
print("=" * 70)
cols19 = set(df19.columns)
cols26 = set(df26.columns)
print(f"In BOTH files     ({len(cols19 & cols26)} cols): {sorted(cols19 & cols26)}")
print(f"\nONLY in 2019 file ({len(cols19 - cols26)} cols): {sorted(cols19 - cols26)}")
print(f"\nONLY in 2026 file ({len(cols26 - cols19)} cols): {sorted(cols26 - cols19)}")

# ── Date coverage ───────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("INSPECTION DATE COVERAGE")
print("=" * 70)
for name, d in [("2019", df19), ("2026", df26)]:
    valid = d["INSPECTION DATE"].dropna()
    valid = valid[valid != pd.Timestamp("1900-01-01")]
    print(f"{name} file: {valid.min().date()} → {valid.max().date()}    "
          f"({len(valid):,} non-placeholder rows)")

print("\nRows per year, side-by-side:")
y19 = df19["INSPECTION DATE"].dt.year.value_counts().sort_index()
y26 = df26["INSPECTION DATE"].dt.year.value_counts().sort_index()
year_compare = pd.concat([y19, y26], axis=1, keys=["2019_file", "2026_file"]).fillna(0).astype(int)
print(year_compare)

# ── Restaurant overlap (by CAMIS) ───────────────────────────────────────────
print("\n" + "=" * 70)
print("RESTAURANT OVERLAP (by CAMIS)")
print("=" * 70)
camis19 = set(df19["CAMIS"].unique())
camis26 = set(df26["CAMIS"].unique())
overlap = camis19 & camis26
only19  = camis19 - camis26
only26  = camis26 - camis19
print(f"Unique CAMIS in 2019 file: {len(camis19):,}")
print(f"Unique CAMIS in 2026 file: {len(camis26):,}")
print(f"Restaurants in BOTH:       {len(overlap):,}    (alive in 2019 AND still alive in 2026)")
print(f"Only in 2019 file:         {len(only19):,}    (likely closed between 2019 and 2026)")
print(f"Only in 2026 file:         {len(only26):,}    (opened after 2019, or were closed in 2019)")

# ── Inspection-type compatibility ───────────────────────────────────────────
print("\n" + "=" * 70)
print("INSPECTION TYPES (top 10) — do they match?")
print("=" * 70)
print("2019 file:")
print(df19["INSPECTION TYPE"].value_counts().head(10))
print("\n2026 file:")
print(df26["INSPECTION TYPE"].value_counts().head(10))

# ── Grade distribution ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("GRADE DISTRIBUTION")
print("=" * 70)
print("2019:", df19["GRADE"].value_counts().to_dict())
print("2026:", df26["GRADE"].value_counts().to_dict())

# ── Cuisine compatibility ──────────────────────────────────────────────────
print("\n" + "=" * 70)
print("CUISINE COMPATIBILITY")
print("=" * 70)
cui19 = set(df19["CUISINE DESCRIPTION"].dropna().unique())
cui26 = set(df26["CUISINE DESCRIPTION"].dropna().unique())
print(f"Cuisines in 2019: {len(cui19)}")
print(f"Cuisines in 2026: {len(cui26)}")
print(f"In both:          {len(cui19 & cui26)}")
print(f"Only in 2019:     {sorted(cui19 - cui26)}")
print(f"Only in 2026:     {sorted(cui26 - cui19)}")

# ── Critical sanity check: gradable inspections per year ────────────────────
print("\n" + "=" * 70)
print("GRADABLE INSPECTIONS PER YEAR (after applying the standard filter)")
print("=" * 70)
GRADABLE = [
    "Cycle Inspection / Initial Inspection",
    "Cycle Inspection / Re-inspection",
    "Pre-permit (Operational) / Initial Inspection",
    "Pre-permit (Operational) / Re-inspection",
]
for name, d in [("2019", df19), ("2026", df26)]:
    f = d[(d["INSPECTION TYPE"].isin(GRADABLE)) &
          (d["GRADE"].isin(["A", "B", "C"])) &
          (d["INSPECTION DATE"] >= "2010-08-01")]
    print(f"\n{name} file → {len(f):,} gradable rows")
    print(f["INSPECTION DATE"].dt.year.value_counts().sort_index().to_string())

BASIC SHAPE
2019 file:   387,301 rows × 26 columns
2026 file:   296,352 rows × 27 columns

COLUMN DIFFERENCES
In BOTH files     (26 cols): ['ACTION', 'BBL', 'BIN', 'BORO', 'BUILDING', 'CAMIS', 'CRITICAL FLAG', 'CUISINE DESCRIPTION', 'Census Tract', 'Community Board', 'Council District', 'DBA', 'GRADE', 'GRADE DATE', 'INSPECTION DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude', 'NTA', 'PHONE', 'RECORD DATE', 'SCORE', 'STREET', 'VIOLATION CODE', 'VIOLATION DESCRIPTION', 'ZIPCODE']

ONLY in 2019 file (0 cols): []

ONLY in 2026 file (1 cols): ['Location']

INSPECTION DATE COVERAGE
2019 file: 2011-10-07 → 2019-08-10    (385,927 non-placeholder rows)
2026 file: 2007-08-10 → 2026-04-23    (292,875 non-placeholder rows)

Rows per year, side-by-side:
                 2019_file  2026_file
INSPECTION DATE                      
1900                  1374       3477
2011                     1          6
2012                     2          6
2013                     9         15
2014               

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# Build 4 dataframes for analysis:
#   df_2026            — 2026 file only, cleaned, full date range
#   df_2026_no_covid   — 2026 file only, COVID grading pause removed
#   df_merged          — 2019 + 2026 merged (2015–2025 coverage)
#   df_merged_no_covid — merged, COVID grading pause removed
#
# Use df_2026* for "current state of NYC dining" cross-sectional analyses.
# Use df_merged* for before/after-COVID comparisons or longitudinal trajectories.
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd

# ── Constants ────────────────────────────────────────────────────────────────
GRADABLE_TYPES = [
    "Cycle Inspection / Initial Inspection",
    "Cycle Inspection / Re-inspection",
    "Pre-permit (Operational) / Initial Inspection",
    "Pre-permit (Operational) / Re-inspection",
]
VALID_GRADES = ["A", "B", "C"]

# Boundary between the two snapshots (max inspection date in the 2019 file)
SPLIT_DATE = pd.Timestamp("2019-08-10")

# COVID grading pause (per DOHMH documentation)
COVID_START = pd.Timestamp("2020-03-17")
COVID_END   = pd.Timestamp("2021-07-19")

# Cuisine label harmonization: 2019 → 2026 conventions
CUISINE_MAP = {
    "Café/Coffee/Tea":                                            "Coffee/Tea",
    "Bakery":                                                     "Bakery Products/Desserts",
    "Ice Cream, Gelato, Yogurt, Ices":                            "Frozen Desserts",
    "Latin (Cuban, Dominican, Puerto Rican, South & Central American)": "Latin American",
    "Soups & Sandwiches":                                         "Soups/Salads/Sandwiches",
    "Steak":                                                      "Steakhouse",
    "Asian":                                                      "Asian/Asian Fusion",
    "Bottled beverages, including water, sodas, juices, etc.":    "Bottled Beverages",
    # "Pizza/Italian" and "Vietnamese/Cambodian/Malaysia" left as-is.
}

# ── Helpers ─────────────────────────────────────────────────────────────────
def clean_snapshot(df, label):
    """Apply the standard cleaning pipeline to one snapshot."""
    print(f"\n── Cleaning {label} ──")
    print(f"   Raw rows: {len(df):,}")

    df = df.copy()
    df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"], errors="coerce")

    df = df[df["INSPECTION DATE"] != pd.Timestamp("1900-01-01")]

    df = df[
        (df["INSPECTION TYPE"].isin(GRADABLE_TYPES)) &
        (df["GRADE"].isin(VALID_GRADES)) &
        (df["INSPECTION DATE"] >= "2011-01-01") &
        (df["BORO"] != "0")
    ]

    df = df[
        (df["INSPECTION DATE"] >= "2011-01-01") &
        (df["INSPECTION DATE"] <= "2025-12-31")
    ]

    print(f"   After cleaning: {len(df):,} rows")
    return df


def assign_period(d):
    if d < COVID_START:
        return "pre_covid"
    if d > COVID_END:
        return "post_covid"
    return "covid_pause"


def drop_covid(df):
    """Remove rows that fall inside the COVID grading pause."""
    return df[~((df["INSPECTION DATE"] >= COVID_START) &
                (df["INSPECTION DATE"] <= COVID_END))].copy()


# ═══════════════════════════════════════════════════════════════════════════
# 1. Load both raw files
# ═══════════════════════════════════════════════════════════════════════════
df19_raw = pd.read_csv("data/NewYorkCity_Restaurant_Inspection_Results_2019.csv",
                       dtype={"ZIPCODE": str}, low_memory=False)
df26_raw = pd.read_csv("data/NewYorkCity_Restaurant_Inspection_Results_2026.csv",
                       dtype={"ZIPCODE": str}, low_memory=False)

# Drop the 2026-only Location column so schemas match
if "Location" in df26_raw.columns:
    df26_raw = df26_raw.drop(columns=["Location"])

# ═══════════════════════════════════════════════════════════════════════════
# 2. Build df_2026 and df_2026_no_covid
# ═══════════════════════════════════════════════════════════════════════════
df_2026 = clean_snapshot(df26_raw, "2026 snapshot")
df_2026["SOURCE"] = "2026_file"
df_2026["PERIOD"] = df_2026["INSPECTION DATE"].apply(assign_period)
df_2026_no_covid = drop_covid(df_2026)

# ═══════════════════════════════════════════════════════════════════════════
# 3. Build df_merged and df_merged_no_covid
# ═══════════════════════════════════════════════════════════════════════════
df_2019 = clean_snapshot(df19_raw, "2019 snapshot")

# Temporal split — 2019 file authoritative ≤ SPLIT_DATE, 2026 file after
print("\n── Applying temporal split for merge ──")
df_2019_for_merge = df_2019[df_2019["INSPECTION DATE"] <= SPLIT_DATE].copy()
df_2026_for_merge = df_2026[df_2026["INSPECTION DATE"] > SPLIT_DATE].copy()
print(f"   2019 file contributes: {len(df_2019_for_merge):,} rows  "
      f"(≤ {SPLIT_DATE.date()})")
print(f"   2026 file contributes: {len(df_2026_for_merge):,} rows  "
      f"(> {SPLIT_DATE.date()})")

# Harmonize cuisine labels on the 2019 portion
df_2019_for_merge["CUISINE DESCRIPTION"] = (
    df_2019_for_merge["CUISINE DESCRIPTION"].replace(CUISINE_MAP)
)

# Tag source
df_2019_for_merge["SOURCE"] = "2019_file"
# (df_2026_for_merge already has SOURCE = "2026_file" from earlier)

# Concatenate
df_merged = pd.concat([df_2019_for_merge, df_2026_for_merge], ignore_index=True)
df_merged["PERIOD"] = df_merged["INSPECTION DATE"].apply(assign_period)
df_merged_no_covid = drop_covid(df_merged)

# ═══════════════════════════════════════════════════════════════════════════
# 4. Verify there are no duplicate inspections in the merged data
# ═══════════════════════════════════════════════════════════════════════════
dupe_key = ["CAMIS", "INSPECTION DATE", "VIOLATION CODE"]
n_dupes = df_merged.duplicated(subset=dupe_key).sum()
mixed = (df_merged.groupby(["CAMIS", "INSPECTION DATE"])["SOURCE"]
                  .nunique().gt(1).sum())
print(f"\n── Duplicate check ──")
print(f"   Duplicate (CAMIS, date, violation) rows in df_merged: {n_dupes:,}")
print(f"   Inspections appearing in BOTH source files: {mixed:,}")
print(f"   (Both should be 0 — confirms the temporal split works.)")

# ═══════════════════════════════════════════════════════════════════════════
# 5. Summary of all 4 dataframes
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("FOUR DATAFRAMES READY FOR ANALYSIS")
print("═" * 70)

def summarize(df, name):
    print(f"\n{name}")
    print(f"   rows:           {len(df):,}")
    print(f"   unique CAMIS:   {df['CAMIS'].nunique():,}")
    print(f"   date range:     {df['INSPECTION DATE'].min().date()} → "
          f"{df['INSPECTION DATE'].max().date()}")
    print(f"   period split:   "
          f"{dict(df['PERIOD'].value_counts())}")

summarize(df_2026,            "df_2026             (2026 file only, full)")
summarize(df_2026_no_covid,   "df_2026_no_covid    (2026 file only, COVID pause removed)")
summarize(df_merged,          "df_merged           (2019 + 2026, full)")
summarize(df_merged_no_covid, "df_merged_no_covid  (2019 + 2026, COVID pause removed)")

print("\n── How to use them ──")
print("   df_2026, df_2026_no_covid → cross-sectional 'NYC right now' analyses")
print("   df_merged, df_merged_no_covid → before/after-COVID, longitudinal trajectories")

# ═══════════════════════════════════════════════════════════════════════════
# Duplicate verification for df_merged
# Run this AFTER the merge script to confirm no inspection appears twice.
# ═══════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("DUPLICATE CHECK ON df_merged")
print("═" * 70)

# ── Check 1: full-row duplicates on (CAMIS, date, violation) ────────────────
# A real duplicate would be the same restaurant with the same inspection
# and the same violation code appearing more than once.
dupe_key = ["CAMIS", "INSPECTION DATE", "VIOLATION CODE"]
n_dupes = df_merged.duplicated(subset=dupe_key).sum()
print(f"\n1. Duplicate (CAMIS, date, violation) rows: {n_dupes:,}")
print(f"   → expected: 0")

# ── Check 2: temporal-split integrity ───────────────────────────────────────
# Did the temporal split actually keep each inspection in exactly one source?
mixed = (df_merged.groupby(["CAMIS", "INSPECTION DATE"])["SOURCE"]
                  .nunique().gt(1).sum())
print(f"\n2. Inspections appearing in BOTH source files: {mixed:,}")
print(f"   → expected: 0  (temporal split should make this impossible)")

# ── Check 3: source × period breakdown (sanity) ─────────────────────────────
print(f"\n3. SOURCE × PERIOD breakdown (where each row comes from):")
print(pd.crosstab(df_merged["SOURCE"], df_merged["PERIOD"]))
print("   → expected: 2019_file is entirely pre_covid; 2026_file is mostly post_covid.")

# ── Check 4: per-restaurant inspection counts ───────────────────────────────
# How many inspections does the typical restaurant have in the merged data?
per_camis = df_merged.groupby("CAMIS").size()
print(f"\n4. Inspections per restaurant in df_merged:")
print(f"   median: {per_camis.median():.0f}")
print(f"   mean:   {per_camis.mean():.1f}")
print(f"   max:    {per_camis.max():,}  ← the busiest restaurant")
print(f"   total unique restaurants: {len(per_camis):,}")

# ── Verdict ─────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
if n_dupes == 0 and mixed == 0:
    print("NO DUPLICATES. df_merged is safe to use for analysis.")
else:
    print("DUPLICATES FOUND — investigate before proceeding.")
print("═" * 70)


── Cleaning 2026 snapshot ──
   Raw rows: 296,352
   After cleaning: 120,310 rows

── Cleaning 2019 snapshot ──
   Raw rows: 387,301
   After cleaning: 185,967 rows

── Applying temporal split for merge ──
   2019 file contributes: 185,967 rows  (≤ 2019-08-10)
   2026 file contributes: 119,161 rows  (> 2019-08-10)

── Duplicate check ──
   Duplicate (CAMIS, date, violation) rows in df_merged: 0
   Inspections appearing in BOTH source files: 0
   (Both should be 0 — confirms the temporal split works.)

══════════════════════════════════════════════════════════════════════
FOUR DATAFRAMES READY FOR ANALYSIS
══════════════════════════════════════════════════════════════════════

df_2026             (2026 file only, full)
   rows:           120,310
   unique CAMIS:   24,104
   date range:     2011-08-10 → 2025-12-31
   period split:   {'post_covid': np.int64(118895), 'pre_covid': np.int64(1415)}

df_2026_no_covid    (2026 file only, COVID pause removed)
   rows:           120,310
   uniqu

In [5]:
SELECTED_CUISINES = [
    "Chinese", "Mexican", "Indian", "Middle Eastern",                # often distrusted / casual
    "Italian", "American", "Mediterranean", "Jewish/Kosher",  # often trusted / upscale
]

---
## 3 Data Analysis



In [14]:
"""
Part W1b — Export Box Plot Stats to JSON for the website
Run AFTER the boxplot cell (uses df_plot and cuisine_categories from it).
Output: boxplot_cuisine.json
"""
import json
import numpy as np

# --- 1. Compute box plot statistics (Tukey 1.5*IQR whiskers, matches seaborn) ---
cuisine_order = cuisine_categories  # from the boxplot cell

stats = {}
for cuisine in cuisine_order:
    scores = df_plot.loc[df_plot['CUISINE DESCRIPTION'] == cuisine, 'SCORE'].values
    q1  = np.percentile(scores, 25)
    med = np.percentile(scores, 50)
    q3  = np.percentile(scores, 75)
    iqr = q3 - q1

    # Tukey whiskers: furthest data point within 1.5 * IQR of the box
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    in_range = scores[(scores >= lower_fence) & (scores <= upper_fence)]
    wmin = in_range.min() if len(in_range) else q1
    wmax = in_range.max() if len(in_range) else q3

    stats[cuisine] = (wmin, q1, med, q3, wmax)

# --- 2. Map each cuisine to a chart CSS variable ---
CHART_VAR_MAP = {
    'American':       'var(--chart-1)',
    'Italian':        'var(--chart-2)',
    'Mediterranean':  'var(--chart-3)',
    'Jewish/Kosher':  'var(--chart-4)',
    'Mexican':        'var(--chart-5)',
    'Middle Eastern': 'var(--chart-6)',
    'Indian':         'var(--chart-1)',
    'Chinese':        'var(--chart-2)',
}

# --- 3. Build the JSON payload ---
payload = []
for cuisine in cuisine_order:
    wmin, q1, med, q3, wmax = stats[cuisine]
    payload.append({
        "name":   cuisine,
        "min":    float(round(wmin, 2)),
        "q1":     float(round(q1,   2)),
        "median": float(round(med,  2)),
        "q3":     float(round(q3,   2)),
        "max":    float(round(wmax, 2)),
        "color":  CHART_VAR_MAP.get(cuisine, 'var(--chart-1)'),
    })

# --- 4. Save to disk ---
with open('boxplot_cuisine.json', 'w') as f:
    json.dump({
        "x_max": 50,
        "scale_label": "Inspection Score (lower = cleaner)",
        "cuisines": payload,
    }, f, indent=2)

print("✓ Saved: boxplot_cuisine.json")
print(json.dumps(payload, indent=2))

✓ Saved: boxplot_cuisine.json
[
  {
    "name": "American",
    "min": 3.0,
    "q1": 9.0,
    "median": 12.0,
    "q3": 13.0,
    "max": 19.0,
    "color": "var(--chart-1)"
  },
  {
    "name": "Italian",
    "min": 6.0,
    "q1": 10.0,
    "median": 12.0,
    "q3": 13.0,
    "max": 17.0,
    "color": "var(--chart-2)"
  },
  {
    "name": "Mediterranean",
    "min": 6.0,
    "q1": 10.0,
    "median": 12.0,
    "q3": 13.0,
    "max": 17.0,
    "color": "var(--chart-3)"
  },
  {
    "name": "Jewish/Kosher",
    "min": 0.0,
    "q1": 10.0,
    "median": 12.0,
    "q3": 20.0,
    "max": 35.0,
    "color": "var(--chart-4)"
  },
  {
    "name": "Mexican",
    "min": 0.0,
    "q1": 10.0,
    "median": 12.0,
    "q3": 19.0,
    "max": 32.0,
    "color": "var(--chart-5)"
  },
  {
    "name": "Middle Eastern",
    "min": 3.0,
    "q1": 10.0,
    "median": 12.0,
    "q3": 15.0,
    "max": 22.0,
    "color": "var(--chart-6)"
  },
  {
    "name": "Indian",
    "min": 0.0,
    "q1": 12.0,
    "medi

In [9]:
"""
Cleveland Dot Plot — one HTML file per violation category
Run after df_2026_no_covid is ready.
Output: 5 separate HTML files, one per category
"""

import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ── 1. Category map ───────────────────────────────────────────────────────────
CATEGORY_MAP = {
    "Personal Hygiene & Food Protection": [
        "06A","06B","06C","06D","06E","06F","06G","06H",
    ],
    "Handling Protocols": [
        "04A","04B","04C","04D","04E","04F","04H","04I","04J","04P",
        "05A","05B","05C","05D","05E","05F","05H","09E","03I","28-05",
    ],
    "Pest Control & Sanitation": [
        "04K","04L","04M","04N","04O","08A","08B","08C","28-06",
    ],
    "Thermal Safety": [
        "02A","02B","02C","02D","02F","02G","02H","02I",
        "03A","03B","03C","03D","03E","03F","03G",
    ],
    "Infrastructure & Maintenance": [
        "07A","09A","09B","09C","09D","10A","10B","10C","10D","10E",
        "10F","10G","10H","10I","10J","28-07",
    ],
}
RAW_COLS = list(CATEGORY_MAP.keys())

SELECTED_CUISINES = [
    "American", "Italian", "Mediterranean", "Jewish/Kosher",
    "Mexican", "Middle Eastern", "Indian", "Chinese",
]

FILE_SLUGS = [
    "cleveland_01_personal_hygiene",
    "cleveland_02_handling_protocols",
    "cleveland_03_pest_control",
    "cleveland_04_thermal_safety",
    "cleveland_05_infrastructure",
]

# ── 2. Build data ─────────────────────────────────────────────────────────────
code_to_cat = {code: cat for cat, codes in CATEGORY_MAP.items() for code in codes}

df_viol = df_2026_no_covid.copy()
df_viol = df_viol[df_viol["CUISINE DESCRIPTION"].isin(SELECTED_CUISINES)]
df_viol = df_viol.dropna(subset=["VIOLATION CODE", "CAMIS"])
df_viol["VIOLATION CODE"] = df_viol["VIOLATION CODE"].str.strip()
df_viol["CATEGORY"] = df_viol["VIOLATION CODE"].map(code_to_cat)
df_viol = df_viol.dropna(subset=["CATEGORY"])

n_restaurants = (
    df_2026_no_covid[df_2026_no_covid["CUISINE DESCRIPTION"].isin(SELECTED_CUISINES)]
    .groupby("CUISINE DESCRIPTION")["CAMIS"].nunique()
)

viol_counts = (
    df_viol.groupby(["CUISINE DESCRIPTION", "CATEGORY"])
    .size().unstack(fill_value=0)
    .reindex(columns=RAW_COLS, fill_value=0)
)

cleveland_df = viol_counts.div(n_restaurants, axis=0).mul(100)

# Consistent order across all panels — cleanest (lowest total) at top
cuisine_order = (
    cleveland_df.sum(axis=1).sort_values(ascending=True).index.tolist()
)
df = cleveland_df.reindex(cuisine_order)

# ── 3. Colors ─────────────────────────────────────────────────────────────────
BG    = '#FAF6F0'
LABEL = '#5C5C5C'
DARK  = '#1A1A1A'
TRACK = '#D4C9BA'
DOT   = '#3D5A73'

# ── 4. One figure per category ────────────────────────────────────────────────
for col, slug in zip(RAW_COLS, FILE_SLUGS):
    values   = df[col].values.astype(float)
    cuisines = cuisine_order
    y_pos    = list(range(len(cuisines)))
    x_max    = float(values.max())
    dtick    = round(x_max / 4, -1) or 10

    fig = go.Figure()

    # Track lines
    for y, v in zip(y_pos, values):
        fig.add_shape(
            type="line",
            x0=0, x1=v, y0=y, y1=y,
            line=dict(color=TRACK, width=1.2),
        )

    # Dots with hover
    fig.add_trace(go.Scatter(
        x=values,
        y=y_pos,
        mode="markers",
        marker=dict(size=10, color=DOT, line=dict(color=BG, width=1.5)),
        customdata=cuisines,
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "%{x:.1f} per 100 restaurants"
            "<extra></extra>"
        ),
        showlegend=False,
    ))

    fig.update_layout(
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        hovermode="closest",
        height=320,
        margin=dict(l=120, r=30, t=50, b=40),

        title=dict(
            text=col,
            x=0, xanchor="left",
            font=dict(family="Lora, Georgia, serif", size=13, color=DARK),
        ),

        xaxis=dict(
            range=[-x_max * 0.03, x_max * 1.05],  # no overshoot
            tickmode="linear",
            tick0=0,
            dtick=dtick,
            tickfont=dict(family="monospace", size=9, color=LABEL),
            title=dict(
                text="Violations per 100 restaurants",
                font=dict(family="Lora, Georgia, serif", size=10, color=LABEL),
            ),
            gridcolor='#E8DFD4',
            gridwidth=0.8,
            zeroline=False,
            showline=False,
            ticklen=0,
        ),

        yaxis=dict(
            tickvals=y_pos,
            ticktext=cuisines,
            range=[-0.7, len(cuisines) - 0.3],
            tickfont=dict(family="Lora, Georgia, serif", size=11, color=LABEL),
            showgrid=False,
            zeroline=False,
            showline=False,
            ticklen=0,
        ),

        hoverlabel=dict(
            bgcolor=DARK,
            font_size=12,
            font_family="monospace",
            font_color=BG,
            bordercolor=DARK,
            align="left",
        ),
    )

    fname = f"{slug}.html"
    fig.write_html(
        fname,
        include_plotlyjs='cdn',
        full_html=True,
        config={"displayModeBar": False, "responsive": True},
    )
    print(f"✓ Saved: {fname}")

✓ Saved: cleveland_01_personal_hygiene.html
✓ Saved: cleveland_02_handling_protocols.html
✓ Saved: cleveland_03_pest_control.html
✓ Saved: cleveland_04_thermal_safety.html
✓ Saved: cleveland_05_infrastructure.html


In [10]:
"""
Export cuisine × year heatmap data from df_merged_no_covid to JSON.

Run this in your notebook (or as a script after the dataframes are built).
Output: heatmap_data.json — drop into your React project.

Structure:
{
  "years":     [2015, 2016, ..., 2025],
  "cuisines":  ["Indian", "Chinese", ...],   # sorted by post-COVID minus pre-COVID delta
  "covid_years": [2020, 2021],
  "data": {
      "Indian": {
          "2015": {"r": 1.28, "s": 16.4, "c": 12.8, "n": 312},
          "2016": {...},
          ...
          "2020": null,
          "2021": null,
          ...
      },
      ...
  }
}
"""

import json
import numpy as np
import pandas as pd

# ── CONFIG ──────────────────────────────────────────────────────────────────
SELECTED_CUISINES = [
    "American", "Italian", "Mediterranean", "Jewish/Kosher",
    "Mexican", "Middle Eastern", "Indian", "Chinese",
]
MIN_INSPECTIONS = 30        # mask cells with fewer than this many inspections
COVID_YEARS = [2020, 2021]  # rendered as the "grading paused" gap
OUTPUT_PATH = "heatmap_data.json"

# Pre/post COVID boundaries for cuisine ordering (use full years, not pause window)
PRE_COVID_END = 2019
POST_COVID_START = 2022

# ── PREP ────────────────────────────────────────────────────────────────────
# Assumes df_merged_no_covid is already in scope from your notebook.
df = df_merged_no_covid.dropna(
    subset=["SCORE", "CUISINE DESCRIPTION", "INSPECTION DATE"]
).copy()

df["SCORE"] = pd.to_numeric(df["SCORE"], errors="coerce")
df = df.dropna(subset=["SCORE"])
df = df[df["SCORE"] >= 0]
df = df[df["CUISINE DESCRIPTION"].isin(SELECTED_CUISINES)]
df["Year"] = df["INSPECTION DATE"].dt.year

# ── AGGREGATE ───────────────────────────────────────────────────────────────
counts = (df.groupby(["CUISINE DESCRIPTION", "Year"])
            .size().rename("Count"))
score_matrix = (df.groupby(["CUISINE DESCRIPTION", "Year"])["SCORE"]
                  .mean().rename("Score"))
city_avg = df.groupby("Year")["SCORE"].mean()

agg = pd.concat([score_matrix, counts], axis=1).reset_index()
agg["CityAvg"] = agg["Year"].map(city_avg)
agg["Ratio"] = agg["Score"] / agg["CityAvg"]

# Mask cells with too few inspections (can't trust the mean)
agg.loc[agg["Count"] < MIN_INSPECTIONS, ["Ratio", "Score"]] = np.nan

# ── CUISINE ORDERING: by post-COVID minus pre-COVID delta ───────────────────
# Cuisines that worsened most appear at the top.
def mean_ratio(sub, year_filter):
    s = sub.loc[year_filter, "Ratio"].dropna()
    return s.mean() if len(s) else np.nan

deltas = {}
for cuisine, sub in agg.groupby("CUISINE DESCRIPTION"):
    sub = sub.set_index("Year")
    pre_mask = sub.index <= PRE_COVID_END
    post_mask = sub.index >= POST_COVID_START
    pre = mean_ratio(sub, pre_mask)
    post = mean_ratio(sub, post_mask)
    deltas[cuisine] = (post - pre) if (np.isfinite(pre) and np.isfinite(post)) else -np.inf

cuisine_order = sorted(SELECTED_CUISINES, key=lambda c: deltas.get(c, -np.inf), reverse=True)

print("Cuisine ordering (post-COVID minus pre-COVID ratio delta):")
for c in cuisine_order:
    d = deltas.get(c, np.nan)
    print(f"  {c:<20} {d:+.3f}")

# ── BUILD JSON-FRIENDLY STRUCTURE ───────────────────────────────────────────
all_years = sorted(int(y) for y in agg["Year"].unique())

# Make sure the COVID years are present in the year list even if no data,
# so the React grid renders the gap cells in the right position.
year_set = set(all_years) | set(COVID_YEARS)
years = sorted(year_set)

data = {}
for cuisine in cuisine_order:
    sub = agg[agg["CUISINE DESCRIPTION"] == cuisine].set_index("Year")
    row = {}
    for y in years:
        if y in COVID_YEARS:
            row[str(y)] = None
            continue
        if y not in sub.index:
            row[str(y)] = None
            continue
        rec = sub.loc[y]
        if pd.isna(rec["Ratio"]):
            row[str(y)] = None
        else:
            row[str(y)] = {
                "r": round(float(rec["Ratio"]), 3),
                "s": round(float(rec["Score"]), 1),
                "c": round(float(rec["CityAvg"]), 1),
                "n": int(rec["Count"]),
            }
    data[cuisine] = row

payload = {
    "years": years,
    "cuisines": cuisine_order,
    "covid_years": COVID_YEARS,
    "data": data,
}

# ── WRITE ───────────────────────────────────────────────────────────────────
with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nWrote {OUTPUT_PATH}")
print(f"  years:      {years[0]}–{years[-1]}  ({len(years)} columns)")
print(f"  cuisines:   {len(cuisine_order)} rows, ordered by post-pre delta")
print(f"  covid gap:  {COVID_YEARS}")

# Quick preview
print("\nPreview (first cuisine, first 4 years):")
first = cuisine_order[0]
for y in years[:4]:
    print(f"  {first} {y}: {data[first][str(y)]}")

Cuisine ordering (post-COVID minus pre-COVID ratio delta):
  Indian               +0.157
  Chinese              +0.079
  Middle Eastern       +0.011
  Mexican              +0.007
  Mediterranean        -0.014
  Italian              -0.023
  Jewish/Kosher        -0.043
  American             -0.058

Wrote heatmap_data.json
  years:      2013–2025  (12 columns)
  cuisines:   8 rows, ordered by post-pre delta
  covid gap:  [2020, 2021]

Preview (first cuisine, first 4 years):
  Indian 2013: None
  Indian 2015: {'r': 1.288, 's': 16.1, 'c': 12.5, 'n': 32}
  Indian 2016: {'r': 1.226, 's': 15.2, 'c': 12.4, 'n': 415}
  Indian 2017: {'r': 1.074, 's': 13.0, 'c': 12.1, 'n': 556}


In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# NYC Borough Map — Choropleth + restaurant dots, with postMessage bridge
# Echoes back requestId so the parent can ignore stale responses
# when filters change quickly.
# ═══════════════════════════════════════════════════════════════════════════

import json
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import plotly.graph_objects as go

# ── 1. Borough geometry ────────────────────────────────────────────────────
df_nta_raw = pd.read_csv("data/NewYorkCity_Map.csv")
df_nta_raw["geometry"] = df_nta_raw["the_geom"].apply(wkt.loads)
gdf_nta = gpd.GeoDataFrame(df_nta_raw, geometry="geometry", crs="EPSG:4326")
gdf_boro = gdf_nta.dissolve(by="BoroName").reset_index()[["BoroName", "geometry"]]

gdf_boro_proj = gdf_boro.to_crs("EPSG:3857")
gdf_boro["centroid_lat"] = gdf_boro_proj.geometry.centroid.to_crs("EPSG:4326").y
gdf_boro["centroid_lon"] = gdf_boro_proj.geometry.centroid.to_crs("EPSG:4326").x

boro_geo = json.loads(gdf_boro.to_json())

BORO_CENTERS = {
    row["BoroName"]: {"lat": float(row["centroid_lat"]),
                      "lon": float(row["centroid_lon"])}
    for _, row in gdf_boro.iterrows()
}
BORO_CENTERS["All Boroughs"] = {"lat": 40.7128, "lon": -73.97}

# ── 2. Restaurant data (latest inspection per CAMIS) ───────────────────────
VALID_BOROS = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
SELECTED_CUISINES = [
    "American", "Italian", "Mediterranean", "Jewish/Kosher",
    "Mexican", "Middle Eastern", "Indian", "Chinese",
]

df = df_2026_no_covid.copy()  # noqa: F821
df["SCORE"]           = pd.to_numeric(df["SCORE"], errors="coerce")
df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"], errors="coerce")
df["BORO"]            = df["BORO"].str.strip()

# Parse European-decimal lat/lon → floats
df["lat"] = pd.to_numeric(
    df["Latitude"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
)
df["lon"] = pd.to_numeric(
    df["Longitude"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
)

df = df[df["BORO"].isin(VALID_BOROS)].dropna(subset=["CAMIS", "SCORE"])
df["SCORE"] = df["SCORE"].astype(int)

df_all = df[df["CUISINE DESCRIPTION"].isin(SELECTED_CUISINES)].copy()
df_all = df_all.sort_values(["CAMIS", "INSPECTION DATE"])
df_latest = df_all.drop_duplicates(subset=["CAMIS"], keep="last").copy()

def infer_grade(g, s):
    if str(g) in ("A", "B", "C"): return str(g)
    return "A" if s < 14 else ("B" if s < 28 else "C")

df_latest["grade"] = df_latest.apply(
    lambda r: infer_grade(r.get("GRADE", ""), r["SCORE"]), axis=1
)
grade_map = df_latest.set_index("CAMIS")["grade"].to_dict()
df_all["grade"] = df_all["CAMIS"].map(grade_map)

# ── 3. Dots: latest-per-restaurant with valid NYC coords ───────────────────
NYC_BBOX = {"lat_min": 40.4, "lat_max": 41.0, "lon_min": -74.3, "lon_max": -73.6}
df_dots = df_latest.dropna(subset=["lat", "lon"]).copy()
df_dots = df_dots[
    df_dots["lat"].between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"]) &
    df_dots["lon"].between(NYC_BBOX["lon_min"], NYC_BBOX["lon_max"])
]
dropped = len(df_latest) - len(df_dots)
print(f"Dots: {len(df_dots):,} restaurants  "
      f"({dropped:,} dropped for missing/invalid coords, "
      f"{dropped/len(df_latest)*100:.1f}%)")

GRADE_COLORS = {"A": "#6b8e3d", "B": "#e8d56d", "C": "#c0392b"}

# ── 4. Initial choropleth stats ────────────────────────────────────────────
def initial_stats():
    stats = (df_latest.groupby("BORO")
             .agg(median_score=("SCORE", "median"),
                  n_restaurants=("CAMIS", "nunique"))
             .reset_index())
    stats["BORO"] = pd.Categorical(stats["BORO"], categories=VALID_BOROS, ordered=True)
    stats = stats.sort_values("BORO").reset_index(drop=True)
    stats["BORO"] = stats["BORO"].astype(str)
    return stats

init_stats = initial_stats()

# ── 5. Figure ──────────────────────────────────────────────────────────────
fig = go.Figure()

# Trace 0: choropleth
fig.add_trace(go.Choroplethmapbox(
    geojson      = boro_geo,
    locations    = init_stats["BORO"],
    featureidkey = "properties.BoroName",
    z            = init_stats["median_score"],
    colorscale   = [[0.00,     "#6b8e3d"],
                    [13/42,    "#a8b86b"],
                    [13.5/42,  "#e8d56d"],
                    [27/42,    "#d49340"],
                    [27.5/42,  "#c0392b"],
                    [1.00,     "#7a1f15"]],
    zmin=0, zmax=42,
    marker=dict(line=dict(color="#FAF6F0", width=2), opacity=0.55),
    colorbar=dict(
        title=dict(text="Median<br>score",
                   font=dict(family="Lora, Georgia, serif", size=11, color="#1A1A1A")),
        tickvals=[6.5, 20.5, 35],
        ticktext=["A (0–13)", "B (14–27)", "C (28+)"],
        tickfont=dict(size=10, color="#1A1A1A"),
        len=0.55, thickness=14, x=1.0, y=0.5, outlinewidth=0,
    ),
    customdata=[[b, 0, None, None, None, None, "—"] for b in VALID_BOROS],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "─────────────<br>"
        "Restaurants: <b>%{customdata[1]}</b><br>"
        "Median score: <b>%{customdata[2]}</b><br>"
        "Grade A: <b>%{customdata[3]}%</b><br>"
        "Grade B: <b>%{customdata[4]}%</b><br>"
        "Grade C: <b>%{customdata[5]}%</b><br>"
        "Top cuisine: <b>%{customdata[6]}</b>"
        "<extra></extra>"
    ),
))

# Trace 1: restaurant dots
dot_colors = df_dots["grade"].map(GRADE_COLORS).tolist()
dot_customdata = df_dots.apply(
    lambda r: [r.get("DBA", "—"), r["CUISINE DESCRIPTION"], r["grade"], int(r["SCORE"])],
    axis=1,
).tolist()

fig.add_trace(go.Scattermapbox(
    lat=df_dots["lat"].tolist(),
    lon=df_dots["lon"].tolist(),
    mode="markers",
    marker=dict(size=6, color=dot_colors, opacity=0.75),
    customdata=dot_customdata,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "%{customdata[1]}<br>"
        "Grade <b>%{customdata[2]}</b> · Score %{customdata[3]}"
        "<extra></extra>"
    ),
    showlegend=False,
    name="restaurants",
))

# Trace 2: borough labels
fig.add_trace(go.Scattermapbox(
    lat=gdf_boro["centroid_lat"], lon=gdf_boro["centroid_lon"],
    mode="text",
    text=gdf_boro["BoroName"].str.upper(),
    textfont=dict(size=13, color="#1A1A1A", family="Lora, Georgia, serif"),
    hoverinfo="skip", showlegend=False,
))

fig.update_layout(
    mapbox=dict(style="carto-positron",
                center=dict(lat=40.7128, lon=-73.97), zoom=9.3),
    paper_bgcolor="#FAF6F0", plot_bgcolor="#FAF6F0",
    margin=dict(l=0, r=0, t=0, b=0),
    height=600,
)

# ── 6. Pre-aggregate records as JSON; filter in JS at runtime ──────────────
records = df_all[["CAMIS", "BORO", "CUISINE DESCRIPTION", "SCORE", "grade"]].rename(
    columns={"CUISINE DESCRIPTION": "cuisine"}
).to_dict(orient="records")

dot_records = df_dots[["CAMIS", "BORO", "CUISINE DESCRIPTION", "SCORE",
                       "grade", "lat", "lon", "DBA"]].rename(
    columns={"CUISINE DESCRIPTION": "cuisine", "DBA": "name"}
).to_dict(orient="records")

payload = {
    "records":     records,
    "dotRecords":  dot_records,
    "boros":       VALID_BOROS,
    "cuisines":    SELECTED_CUISINES,
    "centers":     BORO_CENTERS,
    "gradeColors": GRADE_COLORS,
}

# ── 7. JS bridge ───────────────────────────────────────────────────────────
bridge_js = """
<script>
(function() {
  const PAYLOAD = __PAYLOAD__;
  const BOROS = PAYLOAD.boros;
  const GRADE_COLORS = PAYLOAD.gradeColors;

  // Trace indices: 0 = choropleth, 1 = dots, 2 = labels
  const CHORO_IDX = 0;
  const DOTS_IDX  = 1;

  function getPlot() { return document.querySelector('.plotly-graph-div'); }

  const latestByCamis = {};
  PAYLOAD.records.forEach(r => { latestByCamis[r.CAMIS] = r; });
  const LATEST = Object.values(latestByCamis);
  const ALL = PAYLOAD.records;
  const DOTS = PAYLOAD.dotRecords;

  function median(arr) {
    if (!arr.length) return null;
    const s = [...arr].sort((a,b) => a-b);
    const m = Math.floor(s.length / 2);
    return s.length % 2 ? s[m] : (s[m-1] + s[m]) / 2;
  }

  function applyFilters(rows, { cuisines, grades, borough }) {
    return rows.filter(r => {
      if (borough && borough !== 'All Boroughs' && r.BORO !== borough) return false;
      if (cuisines && cuisines.length && !cuisines.includes(r.cuisine)) return false;
      if (grades && grades.length && !grades.includes(r.grade)) return false;
      return true;
    });
  }

  function computeBoroughStats(filteredLatest) {
    return BOROS.map(b => {
      const rows = filteredLatest.filter(r => r.BORO === b);
      const scores = rows.map(r => r.SCORE);
      const n = new Set(rows.map(r => r.CAMIS)).size;
      const med = median(scores);
      const pct = g => rows.length
        ? Math.round((rows.filter(r => r.grade === g).length / rows.length) * 1000) / 10
        : null;
      const counts = {};
      rows.forEach(r => { counts[r.cuisine] = (counts[r.cuisine]||0) + 1; });
      const top = Object.entries(counts).sort((a,b) => b[1]-a[1])[0];
      return {
        BORO: b, median_score: med, n_restaurants: n,
        pct_A: pct('A'), pct_B: pct('B'), pct_C: pct('C'),
        top_cuisine: top ? top[0] : '—',
      };
    });
  }

  function updateDots(filtered, showDots) {
    const plot = getPlot();
    if (!plot) return;

    if (!showDots) {
      Plotly.restyle(plot, { visible: false }, [DOTS_IDX]);
      return;
    }

    const lat = filtered.map(r => r.lat);
    const lon = filtered.map(r => r.lon);
    const colors = filtered.map(r => GRADE_COLORS[r.grade] || '#888');
    const customdata = filtered.map(r => [r.name || '—', r.cuisine, r.grade, r.SCORE]);

    Plotly.restyle(plot, {
      visible: true,
      lat: [lat],
      lon: [lon],
      'marker.color': [colors],
      customdata: [customdata],
    }, [DOTS_IDX]);
  }

  function update(filters) {
    const filteredLatest = applyFilters(LATEST, filters);
    const filteredAll    = applyFilters(ALL, filters);
    const filteredDots   = applyFilters(DOTS, filters);
    const stats = computeBoroughStats(filteredLatest);

    const plot = getPlot();
    if (!plot) return;

    // Update choropleth
    const z = stats.map(s => s.median_score);
    const customdata = stats.map(s => [
      s.BORO, s.n_restaurants,
      s.median_score == null ? null : Math.round(s.median_score * 10)/10,
      s.pct_A, s.pct_B, s.pct_C, s.top_cuisine,
    ]);
    Plotly.restyle(plot, { z: [z], customdata: [customdata] }, [CHORO_IDX]);

    // Update dots
    updateDots(filteredDots, filters.showDots !== false);

    // Recenter
    const center = PAYLOAD.centers[filters.borough] || PAYLOAD.centers['All Boroughs'];
    const zoom = (filters.borough && filters.borough !== 'All Boroughs') ? 11 : 9.3;
    Plotly.relayout(plot, { 'mapbox.center': center, 'mapbox.zoom': zoom });

    // Report stats back, echoing requestId so parent can drop stale responses
    const allScores = filteredLatest.map(r => r.SCORE);
    const avgScore = allScores.length
      ? Math.round((allScores.reduce((a,b)=>a+b,0) / allScores.length) * 10) / 10
      : null;

    window.parent.postMessage({
      type: 'UPDATE_STATS',
      requestId: filters.requestId,
      total: new Set(filteredLatest.map(r=>r.CAMIS)).size,
      avgScore: avgScore,
      inspections: filteredAll.length,
      dotsShown: filteredDots.length,
    }, '*');
  }

  let lastFilters = {
    borough: 'All Boroughs',
    cuisines: [],
    grades: [],
    showDots: true,
    requestId: undefined,
  };

  window.addEventListener('message', (e) => {
    if (e.data && e.data.type === 'SET_FILTERS') {
      lastFilters = {
        borough:   e.data.borough  || 'All Boroughs',
        cuisines:  e.data.cuisines || [],
        grades:    e.data.grades   || [],
        showDots:  e.data.showDots !== false,
        requestId: e.data.requestId,
      };
      update(lastFilters);
    }
  });

  const waitForPlot = setInterval(() => {
    if (getPlot()) {
      clearInterval(waitForPlot);
      setTimeout(() => {
        update(lastFilters);
        window.parent.postMessage({ type: 'MAP_READY' }, '*');
      }, 200);
    }
  }, 100);
})();
</script>
"""

bridge_js = bridge_js.replace("__PAYLOAD__", json.dumps(payload))

extra_css = """
<style>
  html, body {
    margin: 0; padding: 0;
    background: #FAF6F0;
    overflow: hidden;
    font-family: 'Lora', Georgia, serif;
  }
  .plotly-graph-div { background: #FAF6F0 !important; }
</style>
"""

html = fig.to_html(
    include_plotlyjs="cdn",
    full_html=True,
    div_id="nyc-map",
    config={"displayModeBar": False, "responsive": True},
)
html = html.replace("</head>", extra_css + "</head>")
html = html.replace("</body>", bridge_js + "</body>")

with open("nyc_borough_map.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✓ Saved nyc_borough_map.html")
print(f"  Restaurants (latest): {len(df_latest):,}")
print(f"  Dots plotted:         {len(df_dots):,}")
print(f"  Inspections (all):    {len(df_all):,}")

Dots: 9,028 restaurants  (264 dropped for missing/invalid coords, 2.8%)


C:\Users\guill\AppData\Local\Temp\ipykernel_8412\1525645320.py:101: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Choroplethmapbox(
C:\Users\guill\AppData\Local\Temp\ipykernel_8412\1525645320.py:143: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\guill\AppData\Local\Temp\ipykernel_8412\1525645320.py:160: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


✓ Saved nyc_borough_map.html
  Restaurants (latest): 9,292
  Dots plotted:         9,028
  Inspections (all):    45,399


---
## References

[1] New York City Department of Health and Mental Hygiene. (2026). DOHMH New York City restaurant inspection results [Data set]. NYC OpenData. Retrieved April 30, 2026, from https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data

[2] Tzioumis, K. (2019). New-York-Restaurant-Guide [Computer software]. GitHub. https://github.com/ktzioumis/New-York-Restaurant-Guide/tree/master

[2] Segel, E., & Heer, J. (2010). Narrative visualization: Telling stories with data. IEEE Transactions on Visualization and Computer Graphics, 16(6), 1139–1148. https://doi.org/10.1109/TVCG.2010.179


---
## Use of AI

The technical implementation and presentation of this assignment involved the use of generative AI tools in the following capacities:

* Plotting and Visualization: AI was utilized to assist in refining the plotting code (Matplotlib, Seaborn, and Altair), specifically for optimizing the data-ink ratio, adjusting aesthetic parameters, and troubleshooting complex multi-panel figures.

* Text Refinement: Generative AI was used as a linguistic assistant to verify grammar, improve sentence structure, and ensure clarity throughout the report.

* Verification and Accountability: Every output generated or assisted by AI including code logic, statistical calculations, and text was thoroughly reviewed and verified by the authors. All interpretations, criminological insights, and final conclusions were developed by the authors to ensure the work accurately reflects our analysis of the San Francisco crime dataset.